# Campanhas persistentes de modelos

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="05_treinamento_campanha"


1. Catálogo e orçamento. Chaves hi.pt.demanda representam períodos cadastrados DEMO; não são fontes institucionais. Default: três modelos, máximo de dois processos.

In [ ]:
display(pd.DataFrame(call(cfg,"GET","/v1/training/campaigns")))
from src.data.journey import extract
_,snapshot=extract(cfg,snapshot_id=str(params.snapshot_id) if params.snapshot_id else None)
display(snapshot)
scopes=call(cfg,"GET","/v1/audit/scopes")
display(pd.DataFrame(scopes))
if params.model_count>len(scopes):
    raise ValueError("O catálogo DEMO não possui esse número de séries distintas; use até "+str(len(scopes)))
model_keys=[s["model_key"] for s in scopes[:params.model_count]]
display({"model_count":len(model_keys),"max_parallel_jobs":params.parallel_jobs,"first_keys":model_keys[:5]})


2. Controle opt-in: pause/resume/cancel/retry-failed. Cancelamento é cooperativo; itens em execução concluem, pendentes são cancelados.

In [ ]:
if params.campaign_action not in ("create","observe"):
    params.authorize()
    if not params.campaign_id: raise ValueError("campaign_id obrigatório")
    evidence["control"]=call(cfg,"POST",f"/v1/training/campaigns/{params.campaign_id}/{params.campaign_action}")
    display(evidence["control"])


3. Criar campanha somente quando campaign_action=create, ou observar o ID escolhido. Estado/checkpoints permanecem no PostgreSQL após fechar o notebook.

In [ ]:
if params.campaign_action!="create":
    params=params.model_copy(update={"execute_write":False})
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True,campaign_display_name=params.campaign_display_name)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
    if params.execute_write and state["status"]!="COMPLETED":
        report(NOTEBOOK_ID,evidence,status="FAIL",generation="v3")
        raise RuntimeError("Campanha não completou: "+state["status"]+"; consulte os erros dos jobs persistidos")
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


4. Quantidades comprovadas e progresso. Treinos reais, versões e instâncias de serving são números distintos.

In [ ]:
items=evidence.get("items",[])
display({"requested":len(items),"completed":sum(i["status"]=="COMPLETED" for i in items),"registered_versions":sum(bool(i["model_version"]) for i in items),"automatically_promoted":0,"new_HTTP_servers":0})
display_locations(cfg,items)


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
